# Volume 14, workshop chapter 1: How research works (Python companion)

A research notebook for Erdős's distinct subset sums problem: exhaustive search for the optimal sets, the Conway–Guy construction checked by computer, the second moment bound seen numerically, the odd-cycle lattice of September 2026, and a computer-science application: **subset sum and meet-in-the-middle**, the algorithm that checks $2^n$ sums with about $2^{n/2}$ work. Exercises first, solutions at the end. Requires `numpy`.

In [1]:
import math, itertools, time
import numpy as np
def dissociated(A):
    sums = {0}
    for a in A:
        new = {s + a for s in sums}
        if new & sums:
            return False
        sums |= new
    return True
print(dissociated([3, 5, 6, 7]), dissociated([3, 4, 5, 6]))

True False


## Exercises

**P1.1 (★) The optimal sets.** Write a backtracking search that finds $a(n)$, the least possible largest element of a dissociated $n$-set, for $n\le7$, together with an optimal set. Compare with OEIS A276661: $1,2,4,7,13,24,44,84,161,309$.

**P1.2 (★) Conway–Guy.** Compute the Conway–Guy sets $A_n=\{u_n-u_{n-i}\}$ with $u_{m+1}=2u_m-u_{m-r}$, $r=\mathrm{round}(\sqrt{2m})$, check that they are dissociated for $n\le 22$, and plot $\max A_n/2^n$. Does it look like it converges? To what?

**P1.3 (★★) The second moment, numerically.** For the Conway–Guy set $A_{16}$, compute the exact distribution of $X=\sum\pm a_i$, its variance, and the fraction of values in $|x|<\lambda\sigma$ for several $\lambda$. Compare with Chebyshev's $1-\lambda^{-2}$ and with the normal law. Which inequality in the proof of Theorem 1.4 is the most wasteful?

**P1.4 (★★) The odd cycle.** For $b=3,5,7$, enumerate all $z\in\{-2,\dots,2\}^b$ with $|z_i+\frac12 z_{i-1}|<1$ for all $i$ and check that only $z=0$ survives (Lemma 1.8). Do the same for $b=4,6$ and find the survivors. Check $\det(I+\frac12P)=1+2^{-b}$ numerically.

**P1.5 (★★) A $Q$-dissociated search.** Find all $4$-dissociated triples $u_1<u_2<u_3\le 15$, and for each the dissociated $6$-set of Proposition 1.6. Which has the smallest maximum?

**P1.6 (★★★) CS application: meet in the middle.** *Subset sum* asks whether some subset of a list of integers sums to a target $t$; it is NP-complete (Volume C3), and the naive algorithm checks $2^n$ subsets. Horowitz and Sahni (1974) split the list in two halves, list the $2^{n/2}$ sums of each half, sort them, and look for pairs adding to $t$ with two pointers: time about $2^{n/2}\cdot n$. Implement both, time them for $n=16,20,24$, and use meet in the middle to test whether a set is dissociated (two equal subset sums are a signed relation $\sum c_ia_i=0$, $c_i\in\{-1,0,1\}$: split it in halves). Up to which $n$ can you check the Conway–Guy sets this way?

## Solutions

**P1.1.** Search by increasing maximum $N$: fix $N$ in the set and choose the others below $N$ by backtracking, pruning as soon as a collision appears.

In [2]:
def amin(n):
    N = 1
    while True:
        def rec(chosen, start, sums):
            if len(chosen) == n - 1:
                return chosen
            for a in range(start, N):
                new = {s + a for s in sums}
                if new & sums:
                    continue
                r = rec(chosen + [a], a + 1, sums | new)
                if r is not None:
                    return r
            return None
        r = rec([], 1, {0, N}) if n > 1 else []
        if r is not None:
            return N, sorted(r + [N])
        N += 1
t = time.time()
for n in range(1, 8):
    print(n, amin(n))
print(f"{time.time()-t:.1f} s")

1 (1, [1])
2 (2, [1, 2])
3 (4, [1, 2, 4])
4 (7, [3, 5, 6, 7])
5 (13, [3, 6, 11, 12, 13])
6 (24, [11, 17, 20, 22, 23, 24])


7 (44, [20, 31, 37, 40, 42, 43, 44])
16.3 s


**P1.2.**

In [3]:
u = [0, 1]
for m in range(1, 40):
    r = round(math.sqrt(2 * m)); u.append(2 * u[m] - u[m - r])
def cg(n): return sorted(u[n] - u[n - i] for i in range(1, n + 1))
for n in range(1, 23):
    A = cg(n)
    print(n, A[-1], round(A[-1] / 2**n, 4), dissociated(A))

1 1 0.5 True
2 2 0.5 True
3 4 0.5 True
4 7 0.4375 True
5 13 0.4062 True
6 24 0.375 True
7 44 0.3438 True
8 84 0.3281 True
9 161 0.3145 True
10 309 0.3018 True
11 594 0.29 True
12 1164 0.2842 True
13 2284 0.2788 True
14 4484 0.2737 True
15 8807 0.2688 True
16 17305 0.2641 True
17 34301 0.2617 True
18 68008 0.2594 True


19 134852 0.2572 True


20 267420 0.255 True


21 530356 0.2529 True


22 1051905 0.2508 True


The ratio decreases slowly (it is about $0.2508$ at $n=22$). Bohman's other construction gives $0.22002$, and the September 2026 construction shows that the true $\liminf$ of $a(n)/2^n$ is $0$.

**P1.3.**

In [4]:
A = cg(16)
X = np.array([0])
for a in A:
    X = np.concatenate([X - a, X + a])
sigma = math.sqrt(sum(a * a for a in A))
print("distinct:", len(set(X.tolist())) == 2**16, " sigma/max =", round(sigma / A[-1], 3), " sqrt(n) =", round(math.sqrt(16), 3))
for lam in (0.5, 1, math.sqrt(3), 2, 3):
    frac = (np.abs(X) < lam * sigma).mean()
    normal = math.erf(lam / math.sqrt(2))
    print(f"lambda={lam:.3f}: fraction {frac:.4f}, Chebyshev >= {max(0,1-lam**-2):.4f}, normal {normal:.4f}")

distinct: True  sigma/max = 3.778  sqrt(n) = 4.0
lambda=0.500: fraction 0.3801, Chebyshev >= 0.0000, normal 0.3829
lambda=1.000: fraction 0.6828, Chebyshev >= 0.0000, normal 0.6827
lambda=1.732: fraction 0.9167, Chebyshev >= 0.6667, normal 0.9167
lambda=2.000: fraction 0.9596, Chebyshev >= 0.7500, normal 0.9545
lambda=3.000: fraction 0.9986, Chebyshev >= 0.8889, normal 0.9973


The values are very close to normal, so Chebyshev's inequality is the wasteful step (it gives $0.667$ where the truth is $0.92$). The other loss is $\sigma\le\sqrt n\,a_n$: here $\sigma/a_n$ is well below $\sqrt{16}=4$, since most elements are smaller than the largest. The next chapter fixes the first loss exactly.

**P1.4.**

In [5]:
def survivors(b, R=2):
    out = []
    for z in itertools.product(range(-R, R + 1), repeat=b):
        if all(abs(z[i] + 0.5 * z[i - 1]) < 1 for i in range(b)):
            out.append(z)
    return out
for b in (3, 4, 5, 6, 7):
    s = survivors(b)
    P = np.roll(np.eye(b), 1, axis=0)
    print(b, len(s), [z for z in s if any(z)][:2], round(np.linalg.det(np.eye(b) + 0.5 * P), 6), 1 + (-1)**(b + 1) * 2.0**-b)

3 1 [] 1.125 1.125
4 3 [(-1, 1, -1, 1), (1, -1, 1, -1)] 0.9375 0.9375
5 1 [] 1.03125 1.03125
6 3 [(-1, 1, -1, 1, -1, 1), (1, -1, 1, -1, 1, -1)] 0.984375 0.984375
7 1 [] 1.007812 1.0078125


**P1.5.**

In [6]:
def q_dissociated(us, Q):
    rng_ = range(-(Q - 1), Q)
    return all(sum(c * x for c, x in zip(cs, us)) != 0
               for cs in itertools.product(rng_, repeat=len(us)) if any(cs))
best = []
for u3 in range(3, 16):
    for u2 in range(2, u3):
        for u1 in range(1, u2):
            if q_dissociated((u1, u2, u3), 4):
                S = sorted(x * 2**j for x in (u1, u2, u3) for j in range(2))
                best.append((max(S), (u1, u2, u3), S, dissociated(S)))
for b in sorted(best): print(b)

(28, (10, 13, 14), [10, 13, 14, 20, 26, 28], True)
(30, (7, 13, 15), [7, 13, 14, 15, 26, 30], True)
(30, (11, 14, 15), [11, 14, 15, 22, 28, 30], True)


**P1.6.**

In [7]:
def subset_sum_naive(a, t):
    return any(sum(c) == t for k in range(len(a) + 1) for c in itertools.combinations(a, k))
def sums_of(a):
    s = [0]
    for x in a:
        s = s + [y + x for y in s]
    return s
def subset_sum_mitm(a, t):
    h = len(a) // 2
    L, R = sorted(sums_of(a[:h])), sorted(sums_of(a[h:]))
    i, j = 0, len(R) - 1
    while i < len(L) and j >= 0:
        s = L[i] + R[j]
        if s == t: return True
        if s < t: i += 1
        else: j -= 1
    return False
rng = np.random.default_rng(15)
for n in (16, 20):
    a = [int(x) for x in rng.integers(1, 10**6, n)]; t = sum(a) // 3
    t0 = time.time(); r1 = subset_sum_naive(a, t); t1 = time.time(); r2 = subset_sum_mitm(a, t); t2 = time.time()
    print(n, r1, r2, f"naive {t1-t0:.2f}s, meet-in-the-middle {t2-t1:.4f}s")

def dissociated_mitm(A):
    # A relation sum c_i a_i = 0 with c in {-1,0,1}^n, c != 0, splits as (left signed sum) = -(right signed sum).
    h = len(A) // 2
    def signed(a):
        s = {0: 1}
        for x in a:
            t = {}
            for v, k in s.items():
                for c in (-1, 0, 1):
                    t[v + c * x] = t.get(v + c * x, 0) + 1
            s = t
        return s
    L, R = signed(A[:h]), signed(A[h:])
    if L[0] > 1 or R[0] > 1: return False
    return not any(v != 0 and -v in R for v in L)
for n in (18, 24, 30):
    A = cg(n); t0 = time.time(); r = dissociated_mitm(A)
    print(n, r, f"{time.time()-t0:.2f}s")

16 False False naive 0.02s, meet-in-the-middle 0.0002s


20 False False naive 0.24s, meet-in-the-middle 0.0009s
18 True 0.01s
24 True 0.10s


30 True 3.69s


Meet in the middle checks a signed relation with $3^{n/2}$ work on each side instead of $2^n$ subset sums compared pairwise; this extends the verification of the Conway–Guy sets to $n=30$ in seconds. Bohman's proof that they are dissociated for every $n$ is the only way to go to infinity, and that is the difference between evidence and proof.